# Parallel Colebrook-White Friction Factor Solver with ParFun

The **Colebrook-White equation** is the industry-standard formula for computing the **Darcy-Weisbach friction factor** in turbulent pipe flow:

$$
\frac{1}{\sqrt{f}} = -2 \log_{10}\!\left(\frac{\varepsilon/D}{3.7} + \frac{2.51}{\text{Re}\sqrt{f}}\right)
$$

Because $f$ appears on both sides, there is no closed-form solution -- it must be solved numerically.
We solve it for **500,000 pipe segments** (a large municipal water network) using Brent's method with **20 initial guesses** each, giving **10,000,000 total solves**.

ParFun parallelises this with minimal code changes.

## Program Structure

![Program Structure](Simple_Colebrook_White_Parfun_flowchart.svg)

# Native Python Implementation

In [1]:
import time
from typing import List, Optional

import numpy as np
from scipy.optimize import brentq

np.random.seed(42)

sequential_runtime = None


def colebrook_residual(f: float, Re: float, rel_roughness: float) -> float:
    """Returns g(f) = 0 at the true friction factor."""
    if f <= 0:
        return np.inf
    return 1.0 / np.sqrt(f) + 2.0 * np.log10(rel_roughness / 3.7 + 2.51 / (Re * np.sqrt(f)))


def solve_single_pipe(Re: float, rel_roughness: float, n_guesses: int = 20) -> Optional[float]:
    """Solve Colebrook-White for one pipe segment.

    Sweeps over n_guesses bracketed intervals and accepts the first convergent root.
    Returns the friction factor f, or NaN if no solution found.
    """
    f_candidates = np.logspace(np.log10(0.008), np.log10(0.1), n_guesses + 1)

    for i in range(len(f_candidates) - 1):
        fa = f_candidates[i]
        fb = f_candidates[i + 1]
        ga = colebrook_residual(fa, Re, rel_roughness)
        gb = colebrook_residual(fb, Re, rel_roughness)

        if np.isfinite(ga) and np.isfinite(gb) and ga * gb < 0:
            try:
                root = brentq(colebrook_residual, fa, fb,
                              args=(Re, rel_roughness), xtol=1e-10, maxiter=200)
                return root
            except Exception:
                continue
    return np.nan


def solve_network(
    re_list: List[float],
    roughness_list: List[float],
    n_guesses: int = 20,
) -> List[float]:
    """Solve Colebrook-White for every pipe segment."""
    results = []
    for Re, eps in zip(re_list, roughness_list):
        f = solve_single_pipe(Re, eps, n_guesses)
        results.append(f if f is not None else np.nan)
    return results


def main():
    global sequential_runtime

    N_PIPES = 500_000
    N_GUESSES = 20

    re_values = np.random.uniform(4_000, 2_000_000, N_PIPES).tolist()
    roughness_values = np.random.uniform(0.0001, 0.05, N_PIPES).tolist()

    print(f"Pipeline network: {N_PIPES:,} segments x {N_GUESSES} guesses = {N_PIPES * N_GUESSES:,} solves")

    start = time.time()
    results = solve_network(re_values, roughness_values, N_GUESSES)
    sequential_runtime = time.time() - start

    results_arr = np.array(results)
    n_solved = np.sum(~np.isnan(results_arr))
    print(f"Sequential: {sequential_runtime:.2f}s")
    print(f"Pipes solved: {n_solved:,} / {N_PIPES:,}")


if __name__ == "__main__":
    main()

Pipeline network: 500,000 segments x 20 guesses = 10,000,000 solves
Sequential: 66.46s
Pipes solved: 500,000 / 500,000


# ParFun Implementation

In [2]:
import time
from typing import List, Optional

import numpy as np
from scipy.optimize import brentq

import parfun as pf

np.random.seed(42)

parallel_runtime = None


def colebrook_residual(f: float, Re: float, rel_roughness: float) -> float:
    """Returns g(f) = 0 at the true friction factor."""
    if f <= 0:
        return np.inf
    return 1.0 / np.sqrt(f) + 2.0 * np.log10(rel_roughness / 3.7 + 2.51 / (Re * np.sqrt(f)))


def solve_single_pipe(Re: float, rel_roughness: float, n_guesses: int = 20) -> Optional[float]:
    """Solve Colebrook-White for one pipe segment.

    Sweeps over n_guesses bracketed intervals and accepts the first convergent root.
    Returns the friction factor f, or NaN if no solution found.
    """
    f_candidates = np.logspace(np.log10(0.008), np.log10(0.1), n_guesses + 1)

    for i in range(len(f_candidates) - 1):
        fa = f_candidates[i]
        fb = f_candidates[i + 1]
        ga = colebrook_residual(fa, Re, rel_roughness)
        gb = colebrook_residual(fb, Re, rel_roughness)

        if np.isfinite(ga) and np.isfinite(gb) and ga * gb < 0:
            try:
                root = brentq(colebrook_residual, fa, fb,
                              args=(Re, rel_roughness), xtol=1e-10, maxiter=200)
                return root
            except Exception:
                continue
    return np.nan


@pf.parfun(
    split=pf.per_argument(
        re_list=pf.py_list.by_chunk,
        roughness_list=pf.py_list.by_chunk,
    ),
    combine_with=pf.py_list.concat,
    fixed_partition_size=125_000,
)
def solve_network(
    re_list: List[float],
    roughness_list: List[float],
    n_guesses: int = 20,
) -> List[float]:
    """Solve Colebrook-White for every pipe segment."""
    results = []
    for Re, eps in zip(re_list, roughness_list):
        f = solve_single_pipe(Re, eps, n_guesses)
        results.append(f if f is not None else np.nan)
    return results


def main():
    global parallel_runtime

    N_PIPES = 500_000
    N_GUESSES = 20

    re_values = np.random.uniform(4_000, 2_000_000, N_PIPES).tolist()
    roughness_values = np.random.uniform(0.0001, 0.05, N_PIPES).tolist()

    print(f"Pipeline network: {N_PIPES:,} segments x {N_GUESSES} guesses = {N_PIPES * N_GUESSES:,} solves")

    with pf.set_parallel_backend_context("scaler_local", n_workers=4):
        start = time.time()
        results = solve_network(re_values, roughness_values, N_GUESSES)
        parallel_runtime = time.time() - start

    results_arr = np.array(results)
    n_solved = np.sum(~np.isnan(results_arr))
    print(f"Parallel:   {parallel_runtime:.2f}s")
    print(f"Pipes solved: {n_solved:,} / {N_PIPES:,}")


if __name__ == "__main__":
    main()

Pipeline network: 500,000 segments x 20 guesses = 10,000,000 solves
[INFO]2026-03-31 10:37:59+0800: logging to ('/dev/stdout',)
[INFO]2026-03-31 10:37:59+0800: ObjectStorageServer: start and listen to tcp://127.0.0.1:42171
[INFO]2026-03-31 10:37:59+0800: ObjectStorageServer: started
[INFO]2026-03-31 10:37:59+0800: logging to ('/dev/stdout',)
[INFO]2026-03-31 10:37:59+0800: use event loop: builtin
[INFO]2026-03-31 10:37:59+0800: ConfigController: scheduler_address = tcp://127.0.0.1:36523
[INFO]2026-03-31 10:37:59+0800: ConfigController: object_storage_address = tcp://127.0.0.1:42171
[INFO]2026-03-31 10:37:59+0800: ConfigController: monitor_address = None
[INFO]2026-03-31 10:37:59+0800: ConfigController: protected = True
[INFO]2026-03-31 10:37:59+0800: ConfigController: max_number_of_tasks_waiting = -1
[INFO]2026-03-31 10:37:59+0800: ConfigController: client_timeout_seconds = 60
[INFO]2026-03-31 10:37:59+0800: ConfigController: worker_timeout_seconds = 60
[INFO]2026-03-31 10:37:59+0800: 

# Diff: Native vs ParFun

In [3]:
import re
import difflib
from IPython.display import display, HTML

native_code = In[1]
parfun_code = In[2]

differ = difflib.HtmlDiff(wrapcolumn=90)
table = differ.make_table(
    native_code.splitlines(),
    parfun_code.splitlines(),
    fromdesc="Native Python",
    todesc="With ParFun",
    context=False,
)

# Strip unwanted columns: navigation links, line numbers, and nowrap attribute
table = re.sub(r'<td class="diff_next"[^>]*>.*?</td>', '', table)
table = re.sub(r'<th class="diff_next"[^>]*>.*?</th>', '', table)
table = re.sub(r'<td class="diff_header"[^>]*>.*?</td>', '', table)
table = table.replace('colspan="2"', '')
table = table.replace(' nowrap="nowrap"', '')
table = re.sub(r'(\s*<colgroup></colgroup>){6}',
               '\n        <colgroup></colgroup> <colgroup></colgroup>', table)

css = """<style>
    table.diff { font-family: monospace; font-size: 10px; border-collapse: collapse; width: 100%; }
    table.diff td { padding: 2px 8px; white-space: pre-wrap; word-break: break-all; text-align: left !important; }
    table.diff th { padding: 6px 8px; text-align: center; background-color: #f0f0f0; font-size: 14px; }
    td.diff_add { background-color: #e6ffec; }
    td.diff_chg { background-color: #fff8c5; }
    td.diff_sub { background-color: #ffebe9; }
    span.diff_add { background-color: #ccffd8; }
    span.diff_chg { background-color: #fff2a8; }
    span.diff_sub { background-color: #ffc0c0; }
</style>"""

display(HTML(css + table))

Native Python,With ParFun
import time,import time
"from typing import List, Optional","from typing import List, Optional"
,
import numpy as np,import numpy as np
from scipy.optimize import brentq,from scipy.optimize import brentq
,
,import parfun as pf
,
np.random.seed(42),np.random.seed(42)
,


# Speedup

In [4]:
print(f"Sequential: {sequential_runtime:.2f}s")
print(f"Parallel:   {parallel_runtime:.2f}s")
print(f"Speedup:    {sequential_runtime / parallel_runtime:.2f}x")

Sequential: 66.46s
Parallel:   35.97s
Speedup:    1.85x
